In [1]:
!pip install rapidfuzz requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.7 MB/s eta 0:00:00


In [2]:
# Download word frequency file
import requests

url = "https://raw.githubusercontent.com/kasunw22/sinhala-para-dict/main/word-frequencies/all_counts.txt"
response = requests.get(url)

with open("all_counts.txt", "wb") as f:
    f.write(response.content)

print("Downloaded all_counts.txt")

Downloaded all_counts.txt


In [3]:
# Load the file
with open("all_counts.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines:", len(lines))
print("Sample lines:")
for line in lines[:10]:
    print(line.strip())

Total lines: 2168118
Sample lines:
මේ,2179780
ඒ,1896167
ද,1570866
හා,1470239
බව,1253869
ඇති,1154155
කළ,1084544
කර,1061384
වන,1001760
සඳහා,991826


In [7]:
# Build lexicon
lexicon = set()

for line in lines:
    parts = line.strip().split()
    if len(parts) >= 2:
        sinhala_word = parts[1]
        if freq >= 5:       # ← adjust this threshold if needed
            lexicon.add(word)
    elif len(parts) == 1:
        lexicon.add(parts[0])

print(f"Loaded {len(lexicon)} words into lexicon")
print("Sample words:", list(lexicon)[:10])

Loaded 2168118 words into lexicon
Sample words: ['කළාමෙදින,10', 'දායඋරුවෙල්,3', '‘අචලසමණ’,3', 'බයිට්,696', 'භාවයාශ්\u200dරිත,3', 'ගුරුවරයාණෝ,19', 'ඇළුණ,5', 'දඩුවම්ය,1', 'උදාහරණයක්මෙසෙ,5', 'විඩාදෙන,5']


In [8]:
# Correction function
from rapidfuzz import process, fuzz

def find_closest(word, lexicon_list, score_cutoff=80):
    result = process.extractOne(
        word,
        lexicon_list,
        scorer=fuzz.ratio,
        score_cutoff=score_cutoff
    )
    if result:
        return result[0]   # result[0] is the matched word, result[1] is score
    return word

# Convert to list once for rapidfuzz
lexicon_list = list(lexicon)

def correct_text(noisy_text, score_cutoff=80):
    words = noisy_text.split()
    corrected = []
    for word in words:
        if word in lexicon:
            corrected.append(word)
        else:
            corrected.append(find_closest(word, lexicon_list, score_cutoff))
    return " ".join(corrected)

In [10]:
# Test pipeline
noisy_text = "අයිබවන් ම ප්‍රබත්මට පුළුවනි ොබර සහය වන්න"  # paste STT output here

corrected = correct_text(noisy_text, score_cutoff=80)

print("Original :", noisy_text)
print("Corrected:", corrected)

Original : අයිබවන් ම ප්‍රබත්මට පුළුවනි ොබර සහය වන්න
Corrected: අයිබන්,3 ම ප්‍රබත්මට ටපුළුවනි,4 ොබර සහය වන්න
